In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob

From depth files generated with process_depth.sh script from HPC_analysis/5-reads_distribution

In [ ]:
depth_folder = "."  # change with directory where .depth files are located

In [ ]:
# Obtener todos los archivos .depth
all_depth_files = glob.glob(os.path.join(depth_folder, "*s.depth"))

# Filtrar archivos por prefijos de muestra CV, DC, GB
groups = {"GBM": [], "CV": []}
for f in all_depth_files:
    basename = os.path.basename(f)
    sample_prefix = basename.split("_")[0]
    for g in groups.keys():
        if sample_prefix.startswith(g):
            groups[g].append(f)
            break

In [ ]:
def load_depth(filepath, n):
    df = pd.read_csv(filepath, sep="\t", header=None, names=["Contig", "Position", "Depth"])
    basename = os.path.basename(filepath).replace(".depth", "")
    sample, region = basename.split("_", 1)
    df["Sample"] = "_".join(basename.split("_")[:n])  # junta las dos primeras partes
    df["Region"] = "_".join(basename.split("_")[n:])  # el resto
    return df

In [ ]:
cv_files = glob.glob(os.path.join(depth_folder, "CV*_*s.depth"))
gb_files = glob.glob(os.path.join(depth_folder, "GB*_*s.depth"))

In [ ]:
salmon_folder = "/scratch/mcarazo/ongoing/GB/sRNAseq_may25/salmon_out_EMN_45S_rRNA"
lib_sizes = {}

for f in glob.glob(os.path.join(salmon_folder, "*/quant.sf")):
    sample = os.path.basename(os.path.dirname(f))
    q = pd.read_csv(f, sep="\t")
    print(sample)
    print(q)
    lib_sizes[sample] = q["NumReads"].sum()

In [ ]:
# Cargar datos para cada grupo y concatenar
split_indices = {
    "CV": 3,
    "GBM": 2
}

group_dfs = {}
for g, files in groups.items():
    idx = split_indices.get(g, 2)  # Por defecto, usa 2 si no se especifica
    dfs = [load_depth(f, n=idx) for f in files]
    if dfs:
        df = pd.concat(dfs, ignore_index=True)
        # Normalizar dentro de cada muestra
        df["RelativeDepth"] = df.apply(lambda r: r["Depth"] / lib_sizes.get(r["Sample"], 1),axis=1)
        group_dfs[g] = df

medians = []
for g, df in group_dfs.items():
    # Agrupar por Region y Position para mediana del Depth
    median_df = df.groupby(["Region", "Position"])["RelativeDepth"].median().reset_index()
    median_df["Group"] = g
    medians.append(median_df)

median_data = pd.concat(medians, ignore_index=True)

In [ ]:
regions = sorted(median_data["Region"].unique())
print(regions)

In [ ]:
# Plot con fill_between, mediana por grupo, región por subplot
n_regions = len(regions)
fig, axes = plt.subplots(nrows=n_regions, ncols=1, figsize=(12, 2.5 * n_regions), sharex=False)

if n_regions == 1:
    axes = [axes]

palette = sns.color_palette("Set2") 
group_colors = dict(zip(groups.keys(), palette.as_hex()))

for ax, region in zip(axes, regions):
    for group in groups.keys():
        region_group_data = median_data[(median_data["Region"] == region) & (median_data["Group"] == group)]
        color = "mediumorchid" if group == "GB" else "orange"
        ax.fill_between(region_group_data["Position"], region_group_data["RelativeDepth"],
                color=color, alpha=0.4, label=group)
    ax.set_ylabel("Median Coverage")
    ax.set_title(region, loc="center", fontsize=12)
    ax.set_xlim(median_data[median_data["Region"] == region]["Position"].min(),
                median_data[median_data["Region"] == region]["Position"].max())
    ax.grid(True, linestyle="--", alpha=0.3)
    ax.legend(title="Group")

axes[-1].set_xlabel("Position")
fig.suptitle("Median Coverage per rRNA region by Sample Group", fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# Test estadístico para comparar las distribuciones de coverage en una región específica entre grupos
from scipy.stats import kruskal

for reg in regions:
    data_for_test = []
    for g in groups.keys():
        vals = median_data[(median_data["Region"] == reg) & (median_data["Group"] == g)]["RelativeDepth"].values
        data_for_test.append(vals)

    # Usamos Kruskal-Wallis para comparar medianas entre más de 2 grupos
    stat, pval = kruskal(*data_for_test)
    print(f"Kruskal-Wallis test for region {reg}: stat={stat:.3f}, p-value={pval:.5f}")